# 02 · EDA orientado a decisiones
Señal, estabilidad (adversarial), forma de las relaciones y anomalías. Las 3 figuras clave van a `reports/fig/`.

In [1]:
import os, sys, warnings
warnings.filterwarnings("ignore")
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(ROOT); sys.path.insert(0, ROOT)
import numpy as np, pandas as pd
pd.set_option("display.width", 160)
from src import config as C
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
FIG = Path('reports/fig'); FIG.mkdir(parents=True, exist_ok=True)
from src.de_clean import cargar_base
from src.validation import metrica
df = cargar_base(); tr = df[df.es_test == 0]; te = df[df.es_test == 1]

## 1. Estabilidad: validación adversarial train (oct–nov) vs test (dic)

In [2]:
import lightgbm as lgb
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score
X = pd.concat([tr[tr.t >= 10], te]); ya = X.es_test.values
feats = [c for c in X.columns if c not in C.NO_FEATURES]
p = cross_val_predict(lgb.LGBMClassifier(n_estimators=200, verbose=-1, n_jobs=C.N_JOBS), X[feats], ya, cv=3, method="predict_proba")[:, 1]
print(f"AUC adversarial = {roc_auc_score(ya, p):.3f}  (> 0,7 sería alarma)")

AUC adversarial = 0.443  (> 0,7 sería alarma)


## 2. Anomalía: `dias_ultima_interaccion` es una copia ruidosa de `dias_ultima_transaccion`

In [3]:
eq = tr.dias_ultima_interaccion == tr.dias_ultima_transaccion
ne = tr[~eq]
print(f"iguales: {eq.mean():.1%}")
print(f"Gini de interacción cuando difiere: {metrica(ne.objetivo, -ne.dias_ultima_interaccion):.4f}")
print(f"Gini de transacción en esas mismas filas: {metrica(ne.objetivo, -ne.dias_ultima_transaccion):.4f}")
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(ne.dias_ultima_interaccion, bins=40, color="#888"); ax[0].set_title("Interacción cuando difiere: uniforme 1–364")
for nom, d in [("transacción", tr.dias_ultima_transaccion), ("interacción (≠ transacción)", ne.dias_ultima_interaccion)]:
    s = tr.objetivo if nom == "transacción" else ne.objetivo
    b = pd.qcut(d, 10, labels=False); ax[1].plot(s.groupby(b).mean().values, marker="o", label=nom)
ax[1].set_xlabel("decil de días"); ax[1].set_ylabel("tasa de conversión"); ax[1].legend(); ax[1].set_title("Solo la transacción ordena")
fig.tight_layout(); fig.savefig(FIG / "02_ruido_interaccion.png", dpi=120)

iguales: 54.9%
Gini de interacción cuando difiere: 0.0103
Gini de transacción en esas mismas filas: 0.0884


## 3. Forma: el riesgo alto se 'apaga' a los 180 días sin transacción

In [4]:
b = (tr.dias_ultima_transaccion // 30 * 30).clip(upper=330)
t = tr.groupby(["banda_riesgo", b], observed=True).objetivo.mean().unstack(0)
fig, ax = plt.subplots(figsize=(8, 4))
for c, col in zip(["low", "medium", "high"], ["#2a9d8f", "#e9c46a", "#e76f51"]):
    ax.plot(t.index, t[c], marker="o", label=f"riesgo {c}", color=col)
ax.axvline(180, ls="--", color="k"); ax.text(185, 0.17, "180 días")
ax.set_xlabel("días desde la última transacción"); ax.set_ylabel("tasa de conversión"); ax.legend()
ax.set_title("Riesgo alto: 14 % antes de 180 días, 5 % después")
fig.tight_layout(); fig.savefig(FIG / "02_umbral_180_riesgo_alto.png", dpi=120)
t.round(3).T

dias_ultima_transaccion,0,30,60,90,120,150,180,210,240,270,300,330
banda_riesgo,,,,,,,,,,,,
high,0.159,0.146,0.155,0.145,0.114,0.132,0.055,0.048,0.057,0.048,0.046,0.045
low,0.219,0.209,0.198,0.188,0.187,0.187,0.190,0.182,0.179,0.180,0.163,0.164
medium,0.169,0.156,0.153,0.137,0.143,0.142,0.142,0.128,0.133,0.133,0.131,0.127


## 4. Interacción: riesgo bajo × productos × app × tarjeta

In [5]:
l = tr.copy(); l["perfil"] = np.where(l.activo_movil.eq(1) & l.tiene_tarjeta_credito.eq(1), "app + tarjeta", "otro")
h = l.groupby(["banda_riesgo", "perfil", "numero_productos"], observed=True).objetivo.mean().unstack()
fig, ax = plt.subplots(figsize=(8, 3.6))
im = ax.imshow(h.values, cmap="YlGn", aspect="auto")
ax.set_xticks(range(h.shape[1])); ax.set_xticklabels(h.columns); ax.set_xlabel("número de productos")
ax.set_yticks(range(h.shape[0])); ax.set_yticklabels([f"{a} · {b}" for a, b in h.index])
for i in range(h.shape[0]):
    for j in range(h.shape[1]):
        ax.text(j, i, f"{h.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Tasa de conversión: el cliente bajo riesgo, vinculado y digital convierte 2-3x")
fig.colorbar(im); fig.tight_layout(); fig.savefig(FIG / "02_riesgo_productos_digital.png", dpi=120)
h.round(3)

numero_productos                1      2      3      4      5
banda_riesgo perfil                                          
high         app + tarjeta  0.088  0.079  0.097  0.104  0.123
             otro           0.083  0.086  0.085  0.093  0.092
low          app + tarjeta  0.144  0.258  0.370  0.384  0.392
             otro           0.132  0.134  0.236  0.236  0.256
medium       app + tarjeta  0.134  0.140  0.155  0.170  0.154
             otro           0.138  0.131  0.141  0.148  0.149

## 5. Endeudamiento: castigo desde RDI ≥ 0,6

In [6]:
for th in [0.5, 0.55, 0.6, 0.65]:
    print(th, round(tr[tr.ratio_deuda_ingresos < th].objetivo.mean(), 4), round(tr[tr.ratio_deuda_ingresos >= th].objetivo.mean(), 4))

0.5 0.1524 0.1406
0.55 0.1525 0.1337
0.6 0.1519 0.1314
0.65 0.1514 0.1292


## 6. Variables sin señal (candidatas a quitar)

In [7]:
from src.de_profile import gini_univariado
ruido = ["antiguedad_direccion_meses", "distancia_sucursal_km", "dia_preferido_pago", "tiene_seguro", "es_nuevo_cliente"]
print({c: round(gini_univariado(tr.objetivo, tr[c]), 4) for c in ruido})
print(tr.groupby("dispositivo_principal", observed=True).objetivo.mean().round(3).to_dict())

{'antiguedad_direccion_meses': np.float64(-0.0), 'distancia_sucursal_km': np.float64(0.0031), 'dia_preferido_pago': np.float64(-0.0016), 'tiene_seguro': np.float64(0.0015), 'es_nuevo_cliente': np.float64(-0.0008)}
{'android': 0.151, 'ios': 0.151, 'otro': 0.145, 'web': 0.15}


## Hallazgos
1. **Sin drift**: la validación adversarial no separa octubre–noviembre de diciembre (ver AUC arriba).
2. **`dias_ultima_interaccion` es ruido**: igual a la transacción en el 55 % de las filas y uniforme en el resto (Gini 0,01). Hay que quitarla.
3. **Umbral de 180 días en riesgo alto**: la tasa cae de 0,141 a 0,049.
4. **Riesgo bajo + 3 productos** duplica la tasa (≈0,25); con app y tarjeta llega a 0,37–0,39.
5. **RDI ≥ 0,6** resta ≈2 puntos. Seis columnas más no tienen señal (Gini < 0,005).

## ¿Qué significa para el banco?
El cliente que más convierte es el de **bajo riesgo, ya vinculado y digital**: venta cruzada barata por app. Al de **riesgo alto inactivo hace 6 meses** no conviene llamarlo (5 % de conversión, y además más pérdida esperada). Estas reglas son explicables ante el jurado y ante la SBS.